# Reinforcement learning

In this notebook, I implemented the RL DQN agent and environment. The training environment uses the pre-processed datasets to emulate the market environment to train the DQN model to approximate the Q-Values for the actions. 


## Environment state - observation data

Earlier in my previous notebook, I pre-processed my datasets and normalized their values, I again needed to change the shape of the data for RL training. It is because that the model learns better with context. As described in my previous notebook, the game playing DQN models use at least 3-4 consecutive frames by replacing the RGB channels from the input data. That enabled the model to learn changes or motions within the input data to understand the context. 

In [1]:
# First a function to read a file
import pandas as pd
from pathlib import Path
import os
from tqdm import tqdm

# The folder will be provided as an argument and the function will return the file names
def load_file_names(folder: str) -> pd.DataFrame:

    files = Path(folder).glob("*.parquet")
    # Just to make sure, it returns the same files for both training and validation data
    return tuple((Path('train_dataset')/file.name, Path('validation_dataset')/file.name) for file in files)

In [74]:
file_names = load_file_names("train_dataset")

In [79]:

display("The file names are in pairs: ",file_names[1001])
display("The total file count: " + str(len(file_names)))

print("🌮 Train path:", file_names[1001][0])
print("🌮 Test path:", file_names[1001][1])
print("🌮 Train path:", file_names[1002][0])
print("🌮 Test path:", file_names[1002][1])

'The file names are in pairs: '

(PosixPath('train_dataset/FENC_processed.parquet'),
 PosixPath('validation_dataset/FENC_processed.parquet'))

'The total file count: 3940'

🌮 Train path: train_dataset/FENC_processed.parquet
🌮 Test path: validation_dataset/FENC_processed.parquet
🌮 Train path: train_dataset/RY_processed.parquet
🌮 Test path: validation_dataset/RY_processed.parquet


In [77]:
from sklearn.utils import shuffle
shuffled_file_names = shuffle(file_names, random_state=42)

In [80]:
print("🌮 Shuffled Train path:", shuffled_file_names[1001][0])
print("🌮 Shuffled Test path:", shuffled_file_names[1001][1])
print("🌮 Shuffled Train path:", shuffled_file_names[1002][0])
print("🌮 Shuffled Test path:", shuffled_file_names[1002][1])


🌮 Shuffled Train path: train_dataset/ECCX_processed.parquet
🌮 Shuffled Test path: validation_dataset/ECCX_processed.parquet
🌮 Shuffled Train path: train_dataset/HRL_processed.parquet
🌮 Shuffled Test path: validation_dataset/HRL_processed.parquet


The load_data functions is implemented below. This function is used for changing the data dimensions by grouping them in 60-days of rolling windows. It appends each group (59 previous days + current day) to a new observation dataset. Example, it will convert the original dataset with (n_rows, features) shape to a new (n-60_rows, 60, features) shape. The metadata array will store the useful information about the data for 'Closed price of current day', 'Target Price in 30 days' and 'Date'.  

The Deep RL Course from HuggingFace says that "We stack frames together because it helps us handle the problem of temporal limitation." 

In [ ]:
import numpy as np
    
def load_data(file_path: str, window: int = 60) -> tuple[np.ndarray, np.ndarray]:
    features = ['RSI_norm', 'MACD_line_norm', 'MACD_signal_norm', 'Volume_norm', 'Return']
    X = []
    metadata = []
    try:
        data = pd.read_parquet(file_path, engine='pyarrow')
        # If the data has just a little more than a year of data, I should skip it. 
        # I don't believe such small data will be useful after eliminating the first 60 days. 
        if len(data) < 300:
            return None, None
        for i in range(len(data) - window):
            X.append(data[features].iloc[i:i+window].values)
            # The value should be the 60th period values as I am wrapping up previous 60 days of data.
            # Corrected below to 1 less index to select the correct data
            metadata.append({
                'target': data['Target'].iloc[i+window-1], 
                'date': data.index[i+window-1], 
                'close': data['Close'].iloc[i+window-1]})
        return np.array(X), np.array(metadata)
    except Exception as e:
        print(f"An error occurred while importing data: {e}")
        return None, None
    

In [55]:
temp_data = pd.read_parquet(file_names[104][0], engine='pyarrow')
random_data, random_target = load_data(file_names[104][0])
display("📗 The shape of the data: " + str(temp_data.shape))
display("📙 The shape of the new data: " + str(random_data.shape))

'📗 The shape of the data: (3243, 8)'

'📙 The shape of the new data: (3152, 60, 5)'

In [61]:

print("\n🌸 Shape of random data:", random_data.shape if random_data is not None else "Not Enough data available!")
print("\nSample of the data:", "\n🌼 RSI: ",random_data[0][0][0], "\n🌼 MACD_line: ", random_data[0][0][1], "\n🌼 MACD_signal: ", random_data[0][0][2], "\n🌼 Volume: ", random_data[0][0][3], "\n🌼 Return: ", random_data[0][0][4])
print("\n🌸 Sample of the target metadata:", "\n🌼 Target: ", random_target[0]['target'])


🌸 Shape of random data: (3152, 60, 5)

Sample of the data: 
🌼 RSI:  0.6024939012404252 
🌼 MACD_line:  1.0244439781572843 
🌼 MACD_signal:  0.28872173569971604 
🌼 Volume:  -0.13111844074574705 
🌼 Return:  0.048790143985848466

🌸 Sample of the target metadata: 
🌼 Target:  15.949999809265137


In [65]:
# As I used the 60 days rolling window, the last 60 days were not included in the data. Otherwise, the last 30 days of Target values are NaN in the pre-processed dataset files.
print("The last days of the Target column: ", random_target[-1]['target'])


The last days of the Target column:  10.31999969482422


Checking my hardware accelerator: mps for Apple M CPU, cuda for Nvidia.

In [22]:
import torch
print("Current Accelerator:", torch.accelerator.current_accelerator())
# if torch.accelerator.is_available():
#     tensor = tensor.to(torch.accelerator.current_accelerator())

Current Accelerator: mps


## Reinforcement Learning 

🍋 **What skill should the agent learn?**  
-Predict the stock trend (up, down, flat)

&#x1F34B; **What information does the agent need?**  
-Agent needs today's technical analysis indicators for a given stock as an observation.  

&#x1F34B; **What actions can the agent take?**  
-Discrete choices (up trend - bullish or buy, down trend - bearish or sell, sideways - neutral or hold).   

&#x1F34B; **How do we measure success?**  
-I measure the success by rewards it received from the accurate predictions

&#x1F34B; **When should episodes end?**   
🎠 A single episode is full cycle of processing a stock historical data.  
🎠 Epoch is full cycle of processing all available training data.   
🎠 Training will terminate when it hits the minimum acceptable value.   

#### Plan for reinforcement learning:
**Environment**  
Gymnasium Environment for stock prediction  

**Agent**   
Prediction Agent  

**Actions**  
Buy, Sell, Hold  

**Reward function**  
Calculate the reward for the action by comparing the 'Close' price with 'Target' price of 30 days. 
If the price difference is greater than 10%, it will be assumed as buy or sell. The respective value the will be compared against the action. If the Action matches positive reward 1 will be given, if it doesn't match negative reward 0.5 will be given.  

**State**  
The normalized data values for 'RSI', 'MACD Signal', 'MACD', 'Volume' and 'Price return'. 





In [23]:
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from typing import Optional

The following code is referred from "Unleashing the Power of Reinforcement Learning for Trading" lab by IBM 👉 [Link](https://cognitiveclass.ai/courses/unleashing-the-power-of-reinforcement-learning-for-trading)  
The lab used the Keras and Tensorflow and I am using pytorch library instead. 

load_data(file_names[333][0]) will return whole training data for the selected stock in (n, 60, 5) shape. 


In [ ]:
class MarketEnv(gym.Env):
    def __init__(self):
        # Observation space: 60 time steps × 5 features
        # Shape is (60, 5) for each observation
        self.observation_space = gym.spaces.Box(
            low=-np.inf, # lower and higher bound can be within 1 as my values are normalized. But to be safe, I will keept it to -inf. I think it won't cause any issues
            high=np.inf, 
            shape=(60, 5), 
            dtype=np.float32
        )

        # Prediction is a discrete action: 0 = hold, 1 = buy, 2 = sell
        self.action_space = gym.spaces.Discrete(3)

        # Data and metadata will be set through load_episode_data() method
        self.data = None
        self.metadata = None
        self.current_step = 0
        self.global_step = 0 # To track the total steps across episodes for epsilon decay
        self.total_score = 0
        self.correct_predictions = 0 # For accuracy evaluation
        self.device = torch.device(torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu")
        print(f"Torch Accelerator: {self.device}")
        
    
    ###########################################################

    # def load_episode_data(self, file_path: str, window: int = 60):
    #     # to load the data, I need to pass the file path using the list of filenames indexes. 
    #     self.data, self.metadata = load_data(file_path, window) # returns (n, 60, 5), (m, 60, 5) or None, None
    #     if self.data is None:
    #         return False
    #     return True
    
    # Helper method - translates internal state to observation (referred: Frama custom env doc)
    def _get_obs(self):
        """Convert internal state to observation format.

        self.data[self.current_step] is (60, 5) shape.
        
        Returns:
            Observation of shape (60, 5)
        """
        return torch.from_numpy(self.data[self.current_step].astype(np.float32)).to(self.device)
    
    def _get_info(self):
        """Compute auxiliary information for debugging.

        Returns:
            dict: Information for the episode
        """
        return {
            "score": self.total_score,
            "global_step": self.global_step,
            "correct_predictions": self.correct_predictions,
            "date": self.metadata[self.current_step]['date'],
            "close": self.metadata[self.current_step]['close'],
            "target": self.metadata[self.current_step]['target']
        }
    
    ############################################################

    def reset(self, path: str, window: int = 60, seed: Optional[int] = None):
        """Start a new episode.

        Args:
            seed: Random seed for reproducible episodes
            path: File path to load the episode data

        Returns:
            tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)
        
        self.current_step = 0
        self.total_score = 0
        self.correct_predictions = 0

        self.data, self.metadata = load_data(path, window) # returns (n, 60, 5), (m, 60, 5). When exception occurs will return None, None

        observation = self._get_obs() if self.data is not None else torch.zeros((1, 60, 5), dtype=torch.float32, device=self.device)
        info = self._get_info() if self.metadata is not None else {"score": 0, "correct_predictions": 0, "skip": True}

        return observation, info
    
    
    def step(self, action):
        """Execute one timestep within the environment.

        Args:
            action: The action to take (0-2 for hold, buy, sell)

        Returns:
            tuple: (observation, reward, terminated, truncated, info)
        """
        # Get current metadata values for reward calculation
        meta = self.metadata[self.current_step]
        current_close = meta['close']
        target_price = meta['target']
        
        # Calculate actual trend based on 30-day target
        if target_price is None or np.isnan(target_price):
            # If no target available, end episode
            terminated = True
            reward = 0.0
        else:
            price_return = (target_price - current_close) / current_close
            
            # Determine actual trend: 0=hold, 1=buy (up), 2=sell (down)
            if price_return > 0.10:  # More than 10% increase
                actual_trend = 1  # Buy
            elif price_return < -0.10:  # More than 10% decrease
                actual_trend = 2  # Sell
            else:
                actual_trend = 0  # Hold
            
            # Reward fairly for all predictions may help agent learn better for each actions.
            if action == actual_trend:
                reward = 2
                self.correct_predictions += 1
            elif action == 0 and actual_trend == 1: # Missed the opportunity
                reward = -1
            elif action == 0 and actual_trend == 2: # Unable to avoid loss
                reward = -1
            elif action == 1 and actual_trend == 2: # Incorrect buy before a crash, more severe loss
                reward = -2
            elif action == 2 and actual_trend == 1: # Sold before a rise, missed the opportunity
                reward = -1
            else:
                reward = -1  # Incorrect prediction. Price didn't move any direction.
            
            self.total_score += reward
            
            # Move to next step
            self.current_step += 1
            self.global_step += 1
            
            # Episode should continue unless it makes too many mistakes. This will restict it to run for full episode if it is doing very bad.
            terminated = self.total_score < -500

        # If it reaches the end of the data, it should also terminate the episode as it hit the maximum step.
        truncated = self.current_step >= len(self.data)
        observation = self._get_obs() if not terminated and not truncated else torch.zeros((1, 60, 5), dtype=torch.float32, device=self.device)
        info = self._get_info() if not terminated and not truncated else {"score": self.total_score, "correct_predictions": self.correct_predictions}

        return observation, reward, terminated, truncated, info

In [72]:
env = MarketEnv()
try:
    print("\nEnvironment passes all checks!")
    print("\nObservation space:", env.observation_space, " and it's type is: ", type(env.observation_space))
    print("\nAction space:", env.action_space, " and it's type is: ", type(env.action_space))
except Exception as e:
    print(f"Environment has issues: {e}")

Torch Accelerator: mps

Environment passes all checks!

Observation space: Box(-inf, inf, (60, 5), float32)  and it's type is:  <class 'gymnasium.spaces.box.Box'>

Action space: Discrete(3)  and it's type is:  <class 'gymnasium.spaces.discrete.Discrete'>


The following code for DQNAgent (Deep Q-Network agent). Huggingface Deep RL-Course Unit3 describes that the best idea is to approximate different Q-values for each possible actions in a given state using a parametrized Q-function $Q_\theta(s,a)$ in neural network. And the Q is the Quality of being at that state. 
  
1. **QNetwork**: This is the adaptation from the CleanRL' Atari games DQN implementation. Github Link: [Click](https://docs.cleanrl.dev/rl-algorithms/dqn/#dqn_ataripy)
2. **DQNAgent Initialization**: The agent is initialized with the state and action space, and learning parameters. DQNAgent uses QNetwork to initialize the Action-Value and Target-Action-Value functions.
3. **DQNAgent Remember**: 
4. **DQNAgent Act**: 
5. **DQNAgent Replay**: 

| <img src="assets/DeepRL-HuggingFace.png" width="400" alt="DQN Pseudocode"> |
|:--:| 
| *image credit: [Hugging Face - Deep RL Course](https://huggingface.co/learn/deep-rl-course/unit3/deep-q-algorithm)* |

In [28]:
# Testing here to verify the convolutional layer.
conv1d = nn.Conv1d(5, out_channels=64, kernel_size=5)
result = conv1d(torch.randn(1, 5, 60))
print(result.shape)

torch.Size([1, 64, 56])


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random

# The class implementation is referred from CleanRL DQN implementation for Atari game.  
class QNetwork(nn.Module):
    def __init__(self, env):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(5, out_channels=64, kernel_size=5),
            nn.ReLU(),
            nn.Conv1d(64, out_channels=128, kernel_size=3),
            nn.ReLU(),
            nn.Flatten(),
            nn.Linear(128 * 58, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, env.single_action_space.n),
        )

    def forward(self, x):
        x = x.transpose(1, 2) # Changing the shape from (1, 60, 5) to (1, 5, 60)
        return self.network(x)

In [ ]:
# learned from "Reinforcement Learning and Deep RL Python (Theory and Projects)" course by "AI Sciences", Packt Publishing
# class EpsilonGreedyStrategy:
#     def __init__(self, start: float, end: float, decay: float):
#         self.initial_epsilon = start
#         self.epsilon_decay = decay
#         self.final_epsilon = end

#     def get_exploration_rate(self, current_step: int) -> float:
#         return self.final_epsilon + (self.initial_epsilon - self.final_epsilon) * np.exp(-self.epsilon_decay * current_step)

Tensorboard - [pytorch Doc](https://docs.pytorch.org/docs/main/tensorboard.html)

In [29]:
import wandb
wandb.login()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/toogiib/.netrc
wandb: Currently logged in as: toogii (toogii-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [83]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.tensorboard import SummaryWriter
import time
import numpy as np
from external_library.buffers import ReplayBuffer

In [ ]:
# Copyright notice
#
# This notebook contains code adapted from CleanRL DQN_atari.py
# Copyright (c) 2019 CleanRL developers

# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:

# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.

# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

# Adapted the ClearRL Atari DQN implementation to fit my stock trading environment
class DQNAgent:
    def __init__(
            self, 
            env: gym.Env, 
            state_shape=(60, 5), 
            action_space: int = 3, 
            learning_rate: float = 1e-4,
            initial_epsilon: float = 1.0,
            epsilon_decay: float = 3e-7, # As total rows are around 15million+, to gradually reach 0.01 in the end of the training, I should use around this much of decay. 
            final_epsilon: float = 0.01,
            discount_factor: float = 0.99,
            buffer_size: int = 1000000):

        """
        Initialize the DQN Agent.
        
        Args:
            env: MarketEnv environment
            state_shape: Shape of observation (1, 60, 5)
            action_space: Number of actions (hold, buy, sell)
            learning_rate: How quickly to update Q-values (0-1)
            initial_epsilon: Starting exploration rate (usually 1.0)
            epsilon_decay: How much to reduce epsilon each episode
            final_epsilon: Minimum exploration rate (usually 0.1)
            discount_factor: How much to value future rewards (0-1)
        """
        
        self.action_space = action_space
        self.learning_rate = learning_rate
        self.gamma = discount_factor
        self.initial_epsilon = initial_epsilon
        self.final_epsilon = final_epsilon
        self.epsilon_decay = epsilon_decay
        self.buffer_size = buffer_size
        self.epsilon = initial_epsilon
        self.target_network_frequency = 1000 # Frequency to update target network (in steps)
        self.tau = 0.005
        """tau - the target network update rate"""
          
        self.device = torch.device(torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu")
        print(f"Torch Accelerator: {self.device}")

        self.q_network = QNetwork(env).to(self.device)
        self.optimizer = optim.Adam(self.q_network.parameters(), lr=self.learning_rate)
        self.target_network = QNetwork(env).to(self.device)
        self.target_network.load_state_dict(self.q_network.state_dict())

        self.rb = ReplayBuffer(
            self.buffer_size,
            env.observation_space,
            env.action_space,
            self.device,
            optimize_memory_usage=True,
            handle_timeout_termination=False,
            )
        
    def get_exploration_rate(self, global_step: int) -> float:
        return self.final_epsilon + (self.initial_epsilon - self.final_epsilon) * np.exp(-self.epsilon_decay * global_step)
    
    def remember(self, obs, action, reward, next_obs, terminated):
        """Store experience in replay buffer."""
        self.rb.add(obs, action, reward, next_obs, terminated, info=None)
    
    def act(self, observation, global_step: int):
        """Choose action using epsilon-greedy policy."""
        self.epsilon = self.get_exploration_rate(global_step)
        if np.random.random() < self.epsilon:
            # Explore: random action
            return env.action_space.sample()
        else:
            with torch.no_grad():
                q_values = self.q_network(observation)
            return torch.argmax(q_values, dim=1).cpu().numpy()
    
    def train(self, batch_size):
        """Train the model using random samples from buffer"""
        if self.rb.size < batch_size:
            return
        
        # Sample random batch from replay buffer
        batch = self.rb.sample(batch_size)
        

        with torch.no_grad():
            target_max, _ = self.target_network(batch.next_observations).max(dim=1)
            td_target = batch.rewards.flatten() + self.gamma * target_max * (1 - batch.dones.flatten())
        old_val = self.q_network(batch.observations).gather(1, batch.actions).squeeze()
        # Compute loss and backpropagate
        loss = F.mse_loss(td_target, old_val)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        
        return loss.item()
    
    def update_target_network(self):
        """Update target network weights."""
        for target_network_param, q_network_param in zip(self.target_network.parameters(), self.q_network.parameters()):
                    target_network_param.data.copy_(
                        self.tau * q_network_param.data + (1.0 - self.tau) * target_network_param.data
                    )
    
    
    def save_model(self, path: str):
        """Save model"""
        model_path = path.replace(".pth", f"_{int(time.time())}.pth")
        torch.save(self.q_network.state_dict(), model_path)
        print(f"model saved to {model_path}")
    
    def load_model(self, path: str):
        """Load model"""
        self.q_network.load_state_dict(torch.load(path))
        self.target_network.load_state_dict(self.q_network.state_dict())

The following code is used for estimating the total number of all file samples to run the agent. 
It shows that the training dataset has total of 15,970,289 rows of data, and assuming that my 60 days of rolling window removed the last 60 rows from each files = 236,400 rows. By subtracting that froll all rows will give: 15,733,889 rows of data. 

In [81]:
# calculate the total steps for the Epoch. 
import pyarrow.parquet as pq
total_steps = 0
for train_path, _ in shuffled_file_names:
    try:
        total_steps += pq.read_table(train_path).num_rows
    except Exception as e:
        print(f"Error reading {train_path}: {e}")
print("Total file count: " + str(len(shuffled_file_names)))
print(f"Total steps across all episodes: {total_steps}")

Total file count: 3940
Total steps across all episodes: 15970289


In [ ]:
# Train full list of epoch_files (3940)
def train_agent(env, agent, epoch_files , batch_size=32, update_frequency=10):
    accuracy_counts = [] # This will store the correct_prediction counts for each episode in a list. 
    reward_history = [] # This will store the total reward for each episode in a list.
    start_time = time.time()
    run_name = f"DQN_training_{int(start_time)}"

    wandb.init(
        project="Fin-estimator",
        entity="toogii-personal",
        sync_tensorboard=True,
        config={"epsilon": agent.epsilon, "learning_rate": agent.learning_rate, "epsilon_decay": agent.epsilon_decay, "discount_factor": agent.gamma},
        name=run_name,
        monitor_gym=True,
        save_code=True,
    )
    writer = SummaryWriter(f"runs/{run_name}")
    writer.add_text(
        "hyperparameters",
        "|param|value|\n|-|-|\n", 
        f"| epsilon |{agent.epsilon}|",
        f"| discount_factor |{agent.gamma}|",
        f"| learning_rate |{agent.learning_rate}|",
        f"| epsilon_decay |{agent.epsilon_decay}|",
    )
    for episode in range(len(epoch_files)):
        # The file path for the training data for one episode from epoch_files=(tuple((train_path, val_path),(...)))
        file_path = epoch_files[episode][0]  # training data at index 0, and validation at index 1
        
        try:
            obs, info = env.reset(path=file_path)

            if hasattr(info, "skip") and info["skip"]:
                # print(f"Episode {file_path}: Not enough data.")
                continue
            
            episode_reward = 0.0
            
            while True:
                # Agent takes action
                action = agent.act(obs, env.global_step)
                next_obs, reward, terminated, truncated, info = env.step(action)
                #record rewards for plotting purposes 
                if "final_info" in info:
                    for info in info["final_info"]:
                        if info and "episode" in info:
                            print(f"global_step={env.global_step}, episodic_return={info['episode']['r']}")
                            writer.add_scalar("charts/episodic_return", info["episode"]["r"], env.global_step)
                            writer.add_scalar("charts/episodic_length", info["episode"]["l"], env.global_step)
                # Store experience - save data to reply buffer
                agent.remember(obs, action, reward, next_obs, terminated)
                
                # CRUCIAL step easy to overlook
                obs = next_obs
                
                # Train on batch
                loss = agent.train(batch_size)

                if loss is not None:
                    writer.add_scalar("loss/td_loss", loss, env.global_step)
                
                # As using the soft update method, I am updating the target network at every step.
                agent.update_target_network()
                episode_reward += reward
                
                if terminated or truncated:
                    break
            
            accuracy_counts.append(info["correct_predictions"]) # this will be used for visualizing the improvement.
            reward_history.append(episode_reward)
            print(f"Episode {episode + 1}: Total Reward = {episode_reward:.2f}, Epsilon = {agent.epsilon:.3f}")
        except Exception as e:
            print(f"Error in episode {file_path}: {e}")
            continue
    
    return accuracy_counts, reward_history

In [15]:
# Get all the file paths in tuple of tuples (train, validation)
file_names_list = load_file_names("train_dataset")
file_names_list[0]

(PosixPath('train_dataset/SHW_processed.parquet'),
 PosixPath('validation_dataset/SHW_processed.parquet'))

In [ ]:

agent = DQNAgent(input_shape=(60, 5), action_space=3)
accuracy_counts, reward_history = train_agent(env, agent, epoch_files=file_names_list, batch_size=32)
agent.save_model("dqn_model.pth")

### Backtesting
To assess the performance by exploring how it would have performed using historical data.